# ResNet50 Transfer Learning for Deepfake Audio Detection

This notebook trains a ResNet50-based model for classifying audio spectrograms as real or fake.

## Model Architecture:
- **Base**: ResNet50 pretrained on ImageNet (frozen layers)
- **Head**: Flatten → Dense(512) → Dropout(0.3) → Dense(512) → Dropout(0.3) → Dense(1, sigmoid)
- **Output**: Binary classification (real vs fake)

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

# TensorFlow configuration
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

## Configuration

In [ ]:
# Paths
BASE_DIR = Path('../')
SPECTROGRAM_DIR = BASE_DIR / 'data' / 'spectrograms'
MODEL_DIR = BASE_DIR / 'models' / 'audio' / 'resnet50'

# Training parameters
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 2e-5

# Class names
CLASS_NAMES = ['fake', 'real']

print(f"Spectrogram directory: {SPECTROGRAM_DIR}")
print(f"Model output directory: {MODEL_DIR}")

## Load Dataset

In [ ]:
# Load datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    str(SPECTROGRAM_DIR / 'training'),
    labels='inferred',
    label_mode='binary',
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    str(SPECTROGRAM_DIR / 'validation'),
    labels='inferred',
    label_mode='binary',
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=42
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    str(SPECTROGRAM_DIR / 'testing'),
    labels='inferred',
    label_mode='binary',
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=42
)

print(f"\nClass names: {train_ds.class_names}")

In [ ]:
# Configure dataset for performance
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

## Build ResNet50 Model

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout, Rescaling
from tensorflow.keras.optimizers import RMSprop

# Load ResNet50 base model (pretrained on ImageNet)
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
)

# Freeze base model layers
for layer in base_model.layers:
    layer.trainable = False

print(f"Base model layers: {len(base_model.layers)}")
print(f"Trainable layers: {sum(1 for l in base_model.layers if l.trainable)}")

In [ ]:
# Build model with custom classification head
model = Sequential([
    # Input normalization
    Rescaling(1./255, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    
    # Pretrained ResNet50 base
    base_model,
    
    # Classification head
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.3),
    Dense(512, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
], name='resnet50_deepfake_detector')

model.summary()

In [ ]:
# Compile model
model.compile(
    optimizer=RMSprop(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully!")

## Train Model

In [ ]:
# Define callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-7
    )
]

# Train the model
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

## Training Results

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Training Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('ResNet50 - Training and Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(history.history['loss'], label='Training Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('ResNet50 - Training and Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Evaluate on Test Set

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# Generate predictions for confusion matrix
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Get predictions
y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend((predictions > 0.5).astype(int).flatten())

# Classification report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('ResNet50 - Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## Save Model

In [ ]:
# Create model directory if it doesn't exist
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save as TensorFlow SavedModel format
model.save(str(MODEL_DIR))
print(f"Model saved to: {MODEL_DIR}")

# Verify saved model
loaded_model = tf.keras.models.load_model(str(MODEL_DIR))
print(f"Model loaded successfully!")
print(f"Model input shape: {loaded_model.input_shape}")
print(f"Model output shape: {loaded_model.output_shape}")

## Summary

The ResNet50 model has been trained and saved to `models/audio/resnet50/`.

### Model Details:
- **Base**: ResNet50 pretrained on ImageNet (frozen)
- **Custom head**: Flatten → Dense(512) → Dropout(0.3) → Dense(512) → Dropout(0.3) → Dense(1)
- **Input**: 224x224x3 RGB spectrograms
- **Output**: Binary probability (fake/real)

The model can be loaded for inference using:
```python
model = tf.keras.models.load_model('models/audio/resnet50')
```